# 19. 3D point-cloud and detection — PointNet++, DGCNN, Point Transformer, PointPillars, CenterPoint

Only point counts, channel widths, batch size and BEV grid size are reduced. Hierarchical/block counts and detection objectives are preserved.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 1. PointNet++: three set-abstraction levels

The classification hierarchy keeps FPS + metric-radius ball query + local PointNet + symmetric max pooling at two sampled levels, followed by the global set-abstraction level. Centroid/neighbor counts are reduced tensor lengths.

In [ ]:
def farthest_point_sampling(xyz, count):
    point_count = xyz.size(0)
    selected = torch.zeros(
        count,
        dtype=torch.long,
        device=xyz.device,
    )
    minimum_distance = torch.full(
        (point_count,),
        float("inf"),
        device=xyz.device,
    )
    farthest = torch.tensor(0, device=xyz.device)

    for sample_index in range(count):
        selected[sample_index] = farthest
        centroid = xyz[farthest : farthest + 1]
        distance = (xyz - centroid).square().sum(dim=-1)
        minimum_distance = torch.minimum(
            minimum_distance,
            distance,
        )
        farthest = minimum_distance.argmax()
    return selected


def ball_query(query_xyz, source_xyz, radius, samples):
    distances = torch.cdist(query_xyz, source_xyz)
    groups = []

    for distance_row in distances:
        valid = torch.where(distance_row <= radius)[0]
        if valid.numel() == 0:
            valid = distance_row.argmin().view(1)

        chosen = valid[:samples]
        if chosen.numel() < samples:
            chosen = torch.cat(
                [
                    chosen,
                    chosen[-1:].repeat(samples - chosen.numel()),
                ]
            )
        groups.append(chosen)
    return torch.stack(groups)


class SetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, mlp_dims):
        super().__init__()
        layers = []
        input_dim = 3 + input_feature_dim

        for output_dim in mlp_dims:
            layers.append(nn.Linear(input_dim, output_dim))
            layers.append(nn.ReLU())
            input_dim = output_dim
        self.local_pointnet = nn.Sequential(*layers)

    def forward(
        self,
        source_xyz,
        source_features,
        centroid_count,
        radius,
        neighbors,
    ):
        centroid_ids = farthest_point_sampling(
            source_xyz,
            centroid_count,
        )
        centroid_xyz = source_xyz[centroid_ids]
        neighbor_ids = ball_query(
            centroid_xyz,
            source_xyz,
            radius,
            neighbors,
        )

        relative_xyz = (
            source_xyz[neighbor_ids]
            - centroid_xyz[:, None]
        )
        if source_features is None:
            local_input = relative_xyz
        else:
            local_input = torch.cat(
                [relative_xyz, source_features[neighbor_ids]],
                dim=-1,
            )

        local_features = self.local_pointnet(local_input)
        return centroid_xyz, local_features.max(dim=1).values


class GlobalSetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, mlp_dims):
        super().__init__()
        layers = []
        input_dim = 3 + input_feature_dim
        for output_dim in mlp_dims:
            layers.append(nn.Linear(input_dim, output_dim))
            layers.append(nn.ReLU())
            input_dim = output_dim
        self.pointnet = nn.Sequential(*layers)

    def forward(self, xyz, features):
        centered = xyz - xyz.mean(dim=0, keepdim=True)
        local_input = torch.cat([centered, features], dim=-1)
        return self.pointnet(local_input).max(dim=0).values


class SmallTensorPointNetPlusPlus(nn.Module):
    def __init__(self, classes=4):
        super().__init__()
        self.sa1 = SetAbstraction(0, [8, 8, 16])
        self.sa2 = SetAbstraction(16, [16, 16, 24])
        self.sa3 = GlobalSetAbstraction(24, [24, 32, 48])
        self.classifier = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(24, classes),
        )

    def forward(self, xyz):
        xyz1, features1 = self.sa1(
            xyz,
            None,
            centroid_count=16,
            radius=0.2,
            neighbors=8,
        )
        xyz2, features2 = self.sa2(
            xyz1,
            features1,
            centroid_count=4,
            radius=0.4,
            neighbors=8,
        )
        global_feature = self.sa3(xyz2, features2)
        return self.classifier(global_feature[None])


pointnet_pp = SmallTensorPointNetPlusPlus().to(device)
points = torch.rand(32, 3, device=device)
logits = pointnet_pp(points)
logits.square().mean().backward()

assert hasattr(pointnet_pp, "sa1")
assert hasattr(pointnet_pp, "sa2")
assert hasattr(pointnet_pp, "sa3")
print("PointNet++ logits:", logits.shape)

## 2. DGCNN: four EdgeConv blocks and global aggregation

In [ ]:
def knn_indices(features, k):
    distance = torch.cdist(features, features)
    return distance.topk(
        k=k + 1,
        largest=False,
    ).indices[:, 1:]


class EdgeConvBlock(nn.Module):
    def __init__(self, input_dim, output_dim, k=4):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Linear(2 * input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.LeakyReLU(0.2),
        )

    def forward(self, features):
        neighbor_ids = knn_indices(features.detach(), self.k)
        center = features[:, None].expand(-1, self.k, -1)
        neighbor = features[neighbor_ids]
        edge = torch.cat(
            [center, neighbor - center],
            dim=-1,
        )

        flat = edge.reshape(-1, edge.size(-1))
        encoded = self.mlp(flat).view(
            features.size(0),
            self.k,
            -1,
        )
        return encoded.max(dim=1).values


class SmallTensorDGCNN(nn.Module):
    def __init__(self, classes=4, k=4):
        super().__init__()
        self.edge1 = EdgeConvBlock(3, 8, k)
        self.edge2 = EdgeConvBlock(8, 8, k)
        self.edge3 = EdgeConvBlock(8, 16, k)
        self.edge4 = EdgeConvBlock(16, 24, k)
        self.embedding = nn.Linear(8 + 8 + 16 + 24, 32)
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            nn.Linear(32, classes),
        )

    def forward(self, xyz):
        x1 = self.edge1(xyz)
        x2 = self.edge2(x1)
        x3 = self.edge3(x2)
        x4 = self.edge4(x3)
        local = torch.cat([x1, x2, x3, x4], dim=-1)
        embedded = F.leaky_relu(self.embedding(local), 0.2)
        global_max = embedded.max(dim=0).values
        global_mean = embedded.mean(dim=0)
        return self.classifier(
            torch.cat([global_max, global_mean], dim=-1)[None]
        )


dgcnn = SmallTensorDGCNN().to(device)
dgcnn_logits = dgcnn(points)
assert sum(isinstance(module, EdgeConvBlock) for module in dgcnn.modules()) == 4
print("DGCNN logits:", dgcnn_logits.shape)

## 3. Point Transformer vector attention layer

This keeps the paper's vector attention form: relative-position encoding enters both attention logits and value features; attention is channel-wise and normalized across local neighbors.

In [ ]:
class PointTransformerLayer(nn.Module):
    def __init__(self, channels=16, k=4):
        super().__init__()
        self.k = k
        self.query = nn.Linear(channels, channels)
        self.key = nn.Linear(channels, channels)
        self.value = nn.Linear(channels, channels)
        self.position = nn.Sequential(
            nn.Linear(3, channels),
            nn.ReLU(),
            nn.Linear(channels, channels),
        )
        self.attention = nn.Sequential(
            nn.Linear(channels, channels),
            nn.ReLU(),
            nn.Linear(channels, channels),
        )

    def forward(self, xyz, features):
        neighbor_ids = knn_indices(xyz, self.k)
        relative = xyz[:, None] - xyz[neighbor_ids]
        positional = self.position(relative)

        q = self.query(features)[:, None]
        k = self.key(features)[neighbor_ids]
        v = self.value(features)[neighbor_ids]

        logits = self.attention(q - k + positional)
        weights = logits.softmax(dim=1)
        return (weights * (v + positional)).sum(dim=1)


point_features = nn.Linear(3, 16).to(device)(points)
point_transformer = PointTransformerLayer().to(device)
pt_output = point_transformer(points, point_features)
print("Point Transformer:", pt_output.shape)

## 4. PointPillars PFN: raw + cluster offset + pillar-center offset

In [ ]:
class PillarFeatureNet(nn.Module):
    def __init__(self, output_channels=16, voxel_size=0.5):
        super().__init__()
        self.output_channels = output_channels
        self.voxel_size = voxel_size
        self.point_linear = nn.Linear(9, output_channels, bias=False)
        self.point_norm = nn.BatchNorm1d(output_channels)

    def forward(self, xyz, intensity):
        raw_ids = torch.floor(xyz[:, :2] / self.voxel_size).long()
        minimum_id = raw_ids.min(dim=0).values
        local_ids = raw_ids - minimum_id

        height = int(local_ids[:, 1].max().item()) + 1
        width = int(local_ids[:, 0].max().item()) + 1
        bev = torch.zeros(
            self.output_channels,
            height,
            width,
            device=xyz.device,
        )

        unique_ids = torch.unique(local_ids, dim=0)
        for local_id in unique_ids:
            mask = (local_ids == local_id).all(dim=1)
            pillar_xyz = xyz[mask]
            pillar_intensity = intensity[mask, None]

            cluster_offset = (
                pillar_xyz
                - pillar_xyz.mean(dim=0, keepdim=True)
            )
            raw_id = local_id + minimum_id
            center_xy = (
                raw_id.float() + 0.5
            ) * self.voxel_size
            center_offset_xy = pillar_xyz[:, :2] - center_xy

            augmented = torch.cat(
                [
                    pillar_xyz,
                    pillar_intensity,
                    cluster_offset,
                    center_offset_xy,
                ],
                dim=-1,
            )
            hidden = self.point_linear(augmented)
            hidden = self.point_norm(hidden)
            hidden = F.relu(hidden)
            pillar_feature = hidden.max(dim=0).values

            x_index = int(local_id[0])
            y_index = int(local_id[1])
            bev[:, y_index, x_index] = pillar_feature
        return bev


pillar_encoder = PillarFeatureNet().to(device)
pillar_seeds = torch.rand(16, 3, device=device) * 2.0
lidar_points = pillar_seeds.repeat_interleave(2, dim=0)
lidar_points[1::2, 2] += 0.01
intensity = torch.rand(32, device=device)
bev = pillar_encoder(lidar_points, intensity)
assert pillar_encoder.point_linear.in_features == 9
print("pillar BEV:", bev.shape)

## 5. CenterPoint heads and center-only objective

CenterPoint is center-based in BEV. The heatmap uses the same Gaussian modified focal form; regression heads are supervised only at object centers for offset, z, dimensions and `(sin yaw, cos yaw)`.

In [ ]:
class CenterPointHead(nn.Module):
    def __init__(self, input_channels=16, hidden=24, classes=2):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(input_channels, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(hidden, classes, 1)
        self.offset = nn.Conv2d(hidden, 2, 1)
        self.height = nn.Conv2d(hidden, 1, 1)
        self.dimensions = nn.Conv2d(hidden, 3, 1)
        self.rotation = nn.Conv2d(hidden, 2, 1)
        nn.init.constant_(self.heatmap.bias, -2.19)

    def forward(self, bev):
        hidden = self.backbone(bev)
        return {
            "heatmap_logits": self.heatmap(hidden),
            "offset": self.offset(hidden),
            "height": self.height(hidden),
            "dimensions": self.dimensions(hidden),
            "rotation": self.rotation(hidden),
        }


def gaussian_target(height, width, x, y, radius, device):
    yy, xx = torch.meshgrid(
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing="ij",
    )
    sigma = max((2 * radius + 1) / 6, 1e-3)
    return torch.exp(
        -((xx - x) ** 2 + (yy - y) ** 2)
        / (2 * sigma * sigma)
    )


def modified_focal_loss(logits, target, alpha=2.0, beta=4.0):
    probability = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
    positive = target.eq(1.0)
    negative = target.lt(1.0)
    negative_weight = (1 - target).pow(beta)

    positive_loss = (
        torch.log(probability)
        * (1 - probability).pow(alpha)
        * positive
    )
    negative_loss = (
        torch.log(1 - probability)
        * probability.pow(alpha)
        * negative_weight
        * negative
    )
    count = positive.float().sum().clamp_min(1.0)
    return -(positive_loss.sum() + negative_loss.sum()) / count


def gather_at_centers(feature_map, centers):
    values = []
    for batch_index in range(feature_map.size(0)):
        x = centers[batch_index, :, 0]
        y = centers[batch_index, :, 1]
        values.append(
            feature_map[batch_index, :, y, x].transpose(0, 1)
        )
    return torch.stack(values)


centerpoint = CenterPointHead().to(device)
bev_batch = bev[None]
prediction = centerpoint(bev_batch)
height, width = prediction["heatmap_logits"].shape[-2:]

center = torch.tensor([[[1, 1]]], device=device)
class_id = 1
heatmap_target = torch.zeros_like(prediction["heatmap_logits"])
heatmap_target[0, class_id] = gaussian_target(
    height,
    width,
    x=1,
    y=1,
    radius=1,
    device=device,
)

regression_target = {
    "offset": torch.tensor([[[0.2, 0.3]]], device=device),
    "height": torch.tensor([[[0.5]]], device=device),
    "dimensions": torch.tensor([[[1.5, 2.0, 1.2]]], device=device),
    "rotation": torch.tensor([[[0.0, 1.0]]], device=device),
}

loss = modified_focal_loss(
    prediction["heatmap_logits"],
    heatmap_target,
)
for name, target in regression_target.items():
    predicted = gather_at_centers(prediction[name], center)
    loss = loss + F.l1_loss(predicted, target)
loss.backward()
print("CenterPoint loss:", loss.item())

## 6. CenterPoint local-NMS + multi-class top-K 3D decode

In [ ]:
def decode_centerpoint(prediction, voxel_size=0.5, k=5):
    heatmap = torch.sigmoid(prediction["heatmap_logits"])
    heatmap = heatmap * (
        F.max_pool2d(heatmap, 3, stride=1, padding=1)
        == heatmap
    )

    batch_size, classes, height, width = heatmap.shape
    per_class_k = min(k, height * width)
    class_scores, class_indices = torch.topk(
        heatmap.view(batch_size, classes, -1),
        per_class_k,
        dim=-1,
    )

    final_k = min(k, classes * per_class_k)
    scores, flattened = torch.topk(
        class_scores.reshape(batch_size, -1),
        final_k,
        dim=-1,
    )
    class_ids = torch.div(
        flattened,
        per_class_k,
        rounding_mode="floor",
    )
    spatial_indices = class_indices.reshape(batch_size, -1).gather(
        1,
        flattened,
    )
    y = torch.div(spatial_indices, width, rounding_mode="floor")
    x = spatial_indices % width

    centers = torch.stack([x, y], dim=-1)
    offset = gather_at_centers(prediction["offset"], centers)
    z = gather_at_centers(prediction["height"], centers)
    dimensions = gather_at_centers(prediction["dimensions"], centers)
    rotation = gather_at_centers(prediction["rotation"], centers)

    center_x = (x.float() + offset[..., 0]) * voxel_size
    center_y = (y.float() + offset[..., 1]) * voxel_size
    yaw = torch.atan2(rotation[..., 0], rotation[..., 1])

    boxes = torch.cat(
        [
            center_x[..., None],
            center_y[..., None],
            z,
            dimensions,
            yaw[..., None],
        ],
        dim=-1,
    )
    return scores, class_ids, boxes


scores, classes, boxes = decode_centerpoint(prediction, k=5)
assert boxes.size(-1) == 7
assert boxes.size(1) == min(5, prediction["heatmap_logits"].numel())
print("CenterPoint boxes:", boxes.shape)
print("CenterPoint classes:", classes)

## Structural checklist

PointNet++ now has both sampled SA stages plus global SA; DGCNN has four EdgeConv blocks; Point Transformer uses vector local attention; PointPillars uses all 9 augmented point features; CenterPoint uses Gaussian focal supervision, center-only regression and multi-class top-K decode rather than one global maximum or full-map MSE.